In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

DATA_DIR = Path("afl_datasets")
if not DATA_DIR.exists():
    DATA_DIR = Path("Week 3/Day 1/afl_datasets")

FILES = {
    "players": DATA_DIR / "afl_players_info_raw.csv",
    "player_match": DATA_DIR / "afl_players_round_by_round_stats_raw - afl_players_round_by_round_stats_raw.csv.csv",
    "player_season": DATA_DIR / "afl_players_seasonal_stats_raw.csv",
    "team_match": DATA_DIR / "team_matches_home_away_raw - team_matches_home_away_raw.csv.csv",
}

frames = {name: pd.read_csv(path, low_memory=False) for name, path in FILES.items()}
for name, frame in frames.items():
    print(f"{name:14} rows={len(frame):,} columns={len(frame.columns)}")

print("\nDATA DICTIONARY AND GRAIN")
DATA_DICTIONARY = {
    "players": "One row per player in the player master/profile table. Primary key: id (player identity).",
    "player_match": "One row per player-team-opponent-round-game observation. player_id identifies the player; id is a source row identifier, not a documented match key.",
    "player_season": "One row per player-team-season and finals status. Key candidate: player_id + year + team + is_finals.",
    "team_match": "One row per team perspective of a match. Each game normally appears twice, once for each team, linked by date + teams + round + scores.",
}
for name, description in DATA_DICTIONARY.items():
    print(f"- {name}: {description}")

print("\nJOIN MAP")
print("- players.id (integer) -> player_match.player_id (integer) and player_season.player_id (string after normalization).")
print("- player_match.team/opponent/year/round/match_date -> team_match.team_name/opponent/year/round/match_date after normalizing names and dates.")
print("- No explicit match_id is present in either match-level source; use a canonical composite key and validate it against reciprocal team rows.")

# Normalize identity and time fields without changing raw frames.
players = frames["players"].copy()
player_match = frames["player_match"].copy()
player_season = frames["player_season"].copy()
team_match = frames["team_match"].copy()

for frame in (player_match, team_match):
    frame["match_date"] = pd.to_datetime(frame["match_date"], errors="coerce")

players["player_id_key"] = players["id"].astype("string")
player_match["player_id_key"] = player_match["player_id"].astype("string")
player_season["player_id_key"] = player_season["player_id"].astype("string")

for frame, columns in ((player_match, ["team", "opponent"]), (team_match, ["team_name", "opponent"])):
    for column in columns:
        frame[f"{column}_key"] = (frame[column].astype("string").str.strip().str.replace(r"\\s+", " ", regex=True))

player_match["match_key"] = (player_match["match_date"].dt.strftime("%Y-%m-%d") + "|" + player_match["team_key"] + "|" + player_match["opponent_key"] + "|" + player_match["round"].astype("string"))
team_match["match_key"] = (team_match["match_date"].dt.strftime("%Y-%m-%d") + "|" + team_match["team_name_key"] + "|" + team_match["opponent_key"] + "|" + team_match["round"].astype("string"))

print("\nCOVERAGE")
coverage = {
    "players": {"date_min": players["born_date"].min(), "date_max": players["last_date"].max(), "seasons": None, "teams": players["player_teams"].nunique(), "players": players["player_id_key"].nunique()},
    "player_match": {"date_min": player_match["match_date"].min(), "date_max": player_match["match_date"].max(), "seasons": player_match["year"].nunique(), "teams": player_match["team_key"].nunique(), "players": player_match["player_id_key"].nunique()},
    "player_season": {"date_min": None, "date_max": None, "seasons": player_season["year"].nunique(), "teams": player_season["team"].nunique(), "players": player_season["player_id_key"].nunique()},
    "team_match": {"date_min": team_match["match_date"].min(), "date_max": team_match["match_date"].max(), "seasons": team_match["year"].nunique(), "teams": team_match["team_name_key"].nunique(), "players": None},
}
for name, values in coverage.items():
    print(name, values)
print("Observed match-stat seasons:", sorted(player_match["year"].dropna().unique().tolist()))
print("Observed team-match seasons:", sorted(team_match["year"].dropna().unique().tolist()))

print("\nDUPLICATES AND MISSINGNESS")
def quality_report(name, frame, key_columns):
    duplicate_rows = int(frame.duplicated().sum())
    duplicate_keys = int(frame.duplicated(key_columns, keep=False).sum())
    missing = (frame.isna().mean().mul(100).sort_values(ascending=False).head(12).round(2).to_dict())
    print(f"{name}: exact_duplicate_rows={duplicate_rows:,}; duplicate_key_rows={duplicate_keys:,}; key={key_columns}")
    print("  highest_missing_percent:", missing)

quality_report("players", players, ["id"])
quality_report("player_match", player_match, ["player_id_key", "match_key"])
quality_report("player_season", player_season, ["player_id_key", "year", "team", "is_finals"])
quality_report("team_match", team_match, ["match_key"])

print("\nIDENTITY AND JOIN CHECKS")
unknown_player_match = sorted(set(player_match["player_id_key"].dropna()) - set(players["player_id_key"].dropna()))
unknown_player_season = sorted(set(player_season["player_id_key"].dropna()) - set(players["player_id_key"].dropna()))
print("player_match IDs absent from player master:", len(unknown_player_match), unknown_player_match[:10])
print("player_season IDs absent from player master:", len(unknown_player_season), unknown_player_season[:10])

match_name_sets = {
    "player_match teams": set(player_match["team_key"].dropna()),
    "player_match opponents": set(player_match["opponent_key"].dropna()),
    "team_match teams": set(team_match["team_name_key"].dropna()),
    "team_match opponents": set(team_match["opponent_key"].dropna()),
}
all_match_names = sorted(set().union(*match_name_sets.values()))
print("unique team-name tokens across match tables:", len(all_match_names))
print("names only in player-match tables:", sorted((match_name_sets["player_match teams"] | match_name_sets["player_match opponents"]) - (match_name_sets["team_match teams"] | match_name_sets["team_match opponents"])))
print("names only in team-match tables:", sorted((match_name_sets["team_match teams"] | match_name_sets["team_match opponents"]) - (match_name_sets["player_match teams"] | match_name_sets["player_match opponents"])))

# A player-match observation can be linked to team-match rows only when the composite key exists.
player_match_key_coverage = player_match["match_key"].isin(set(team_match["match_key"]))
print(f"player-match rows with an exact composite-key team-match counterpart: {player_match_key_coverage.mean():.1%}")
print(f"team-match rows with an exact composite-key player-match counterpart: {team_match['match_key'].isin(set(player_match['match_key'])).mean():.1%}")
print("team-match rows with reciprocal opponent perspective:", team_match["match_key"].isin(set(team_match["match_date"].dt.strftime("%Y-%m-%d") + "|" + team_match["opponent_key"] + "|" + team_match["team_name_key"] + "|" + team_match["round"].astype("string"))).mean())

print("\nSTRUCTURAL CHANGE SIGNALS")
metric_columns = ["goals", "behinds", "hit_outs", "tackles", "rebound_50s", "inside_50s", "clearances", "brownlow_votes", "contested_possessions", "goal_assist", "percentage_of_game_played"]
metric_presence = player_match.groupby("year")[metric_columns].apply(lambda x: x.notna().mean()).round(3)
print("player-match metric availability by season (first/last five seasons):")
print(pd.concat([metric_presence.head(5), metric_presence.tail(5)]).to_string())
team_counts = team_match.groupby("year")["team_name_key"].nunique()
print("team counts by season (first/last five):")
print(pd.concat([team_counts.head(5), team_counts.tail(5)]).to_string())
print("Team-name changes to review:", sorted({name for name in all_match_names if any(token in name.lower() for token in ["kangaroo", "bulldog", "bear", "swan", "power", "giant", "sun", "crows"])}))

print("\nOUTLIER AND DOMAIN CHECKS")
nonnegative_stats = ["kicks", "marks", "handballs", "disposals", "goals", "behinds", "hit_outs", "tackles", "rebound_50s", "inside_50s", "clearances", "clangers", "free_kicks_for", "free_kicks_against", "brownlow_votes", "contested_possessions", "uncontested_possessions", "contested_marks", "marks_inside_50", "one_percenters", "bounces", "goal_assist", "fantasy_points"]
for column in nonnegative_stats:
    if column in player_match:
        values = pd.to_numeric(player_match[column], errors="coerce")
        negative_count = int((values < 0).sum())
        q99 = values.quantile(.99)
        max_value = values.max()
        print(f"{column:24} negative={negative_count:4} p99={q99:8.1f} max={max_value:8.1f}")

print("disposals != kicks + handballs:", int((player_match["disposals"].notna() & player_match["kicks"].notna() & player_match["handballs"].notna() & (player_match["disposals"] != player_match["kicks"] + player_match["handballs"])).sum()))
print("team score reconciliation failures:", int((team_match["team_score"] != team_match["team_goals_kicked"] * 6 + team_match["team_behinds"]).sum()))
print("team margin reconciliation failures:", int((team_match["margin"].abs() != (team_match["team_score"] - team_match["opponent_score"]).abs()).sum()))
print("invalid result labels:", sorted(set(team_match["result"].dropna()) - {"W", "L", "D"}))

print("\nMODELING DEFINITIONS TO CARRY FORWARD")
print("- Team win: team-match result == 'W'; derive a one-row-per-game target after deduplicating reciprocal team perspectives. Treat D as non-win, not a loss.")
print("- Top player: report multiple explicit leaderboards rather than one ambiguous label: season total goals, goals per game, disposals per game, fantasy points per game, and Brownlow votes. Apply a minimum games threshold before ranking rates.")
print("- Leakage control: only use information available before the prediction match; do not use same-match result, margin, score, or post-match player totals as features.")
print("- Join caution: no source match_id is supplied. Keep the composite key and its coverage report with every downstream feature table.")


players        rows=2,848 columns=16
player_match   rows=274,089 columns=36
player_season  rows=25,491 columns=54
team_match     rows=15,808 columns=19

DATA DICTIONARY AND GRAIN
- players: One row per player in the player master/profile table. Primary key: id (player identity).
- player_match: One row per player-team-opponent-round-game observation. player_id identifies the player; id is a source row identifier, not a documented match key.
- player_season: One row per player-team-season and finals status. Key candidate: player_id + year + team + is_finals.
- team_match: One row per team perspective of a match. Each game normally appears twice, once for each team, linked by date + teams + round + scores.

JOIN MAP
- players.id (integer) -> player_match.player_id (integer) and player_season.player_id (string after normalization).
- player_match.team/opponent/year/round/match_date -> team_match.team_name/opponent/year/round/match_date after normalizing names and dates.
- No explicit matc

## Task 2: Define prediction targets precisely

### Decision for match winner
We will define the primary match-level target as a classification problem at the team-game level:

- `team_win_flag = 1` when `result == "W"`
- `team_win_flag = 0` when `result == "L"`
- `team_win_flag = 0.5` for `result == "D"` only if we want a soft target for a probabilistic model; otherwise keep draws in a separate `result_class` label and exclude them from the binary baseline

Why classification first?
- The core business question is “who wins this match?”
- The dataset already stores one row per team perspective of a match
- A binary classification target is easier to interpret and easier to calibrate into winning probabilities
- `team_margin` remains useful as a secondary regression target for modeling score spread and for ranking performance, but it is not the primary business target

### Definitions for top player targets
We define at least two explicit player-level labels, not one vague “best player” metric:

1. `top_disposal_getter`: player with highest total disposals in a season or per-game disposals
2. `top_goal_kicker`: player with highest total goals in a season or per-game goals
3. `fantasy_value_score`: a composite per-game value metric using the existing fantasy points field

Composite metric formula:

- `fantasy_per_game = fantasy_points / games_played`

This is a valid composite metric because it combines weighted in-game contributions into one score per game while controlling for variation in matches played.

### Target contract for Day 2
The following table becomes the contract for the modeling work.


### Target contract table

| target_name | definition | formula | level of aggregation |
|---|---|---|---|
| `team_win_flag` | Binary win indicator for a team in a match | `1` if `result == "W"`, `0` if `result == "L"`; optional draw handling `0.5` for `"D"` | team-game |
| `team_margin` | Point difference in a match | `team_score - opponent_score` | team-game |
| `top_disposal_getter` | Player with highest disposal volume | `total_disposals` or `disposals_per_game = disposals / games_played` | player-season or player-game rate |
| `top_goal_kicker` | Player with highest goal count | `total_goals` or `goals_per_game = goals / games_played` | player-season or player-game rate |
| `fantasy_value_score` | Composite value score for player contribution | `fantasy_per_game = fantasy_points / games_played` | player-game average |
| `brownlow_style_score` | Reward-based performance score | `brownlow_votes` or `brownlow_votes_per_game` | player-season or player-game |

### Recommendation for Day 2
- Primary classification target: `team_win_flag`
- Secondary regression target: `team_margin`
- Primary player targets: `top_goal_kicker`, `top_disposal_getter`
- Composite player metric: `fantasy_per_game`

This framing keeps the modeling contract explicit and avoids vague labels like “best player.” The actual labels should be defined by season or by match window depending on the task, and all rate-based metrics should use a minimum games threshold to avoid inflated rankings from short samples.


## Task 3: Exploratory Data Analysis

### Team-level EDA
We will inspect:
- win rate over time by team and overall league trend
- home-ground advantage by `home_away`
- season ladder trends using points and win percentage
- recent form vs win probability
- rest-gap effects

### Player-level EDA
We will analyze:
- historical leaders in goals, disposals, and fantasy points
- per-game distribution and consistency of these metrics
- proxy role patterns based on stat profiles because the raw data does not include an explicit `position` field

### Important caveat
The match tables provide strong team-level signals such as home/away, date, result, and margin, but they do not include explicit weather, travel distance, or player position fields. For those cases, we will use valid proxies available in the data and note any unavailable variables explicitly.

### Planned relationships to visualize
1. `team win rate by season`
2. `home_away win rate`
3. `recent form vs win probability`
4. `rest days between matches vs win probability`
5. `distribution of goals, disposals, fantasy points`
6. `historical leaderboard for top goal-kickers and disposal-getters`
7. `role-style signal` based on stat profile proxy

### Why these EDA views matter
These are directly relevant to the end goals:
- Team-level signals inform the match winner model
- Player-level distributions define the “top player” label more rigorously
- The EDA also gives the chat assistant grounded historical context it can explain in natural language



In [ ]:
# ============================================================
# TASK 3: EXPLORATORY DATA ANALYSIS
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. Load datasets
# ------------------------------------------------------------

DATA_DIR = Path("afl_datasets")

if not DATA_DIR.exists():
    DATA_DIR = Path("Week 3/Day 1/afl_datasets")

team_match = pd.read_csv(
    DATA_DIR / "team_matches_home_away_raw - team_matches_home_away_raw.csv.csv",
    low_memory=False
)

player_match = pd.read_csv(
    DATA_DIR / "afl_players_round_by_round_stats_raw - afl_players_round_by_round_stats_raw.csv.csv",
    low_memory=False
)

player_season = pd.read_csv(
    DATA_DIR / "afl_players_seasonal_stats_raw.csv",
    low_memory=False
)

# Convert dates
team_match["match_date"] = pd.to_datetime(
    team_match["match_date"], errors="coerce"
)

player_match["match_date"] = pd.to_datetime(
    player_match["match_date"], errors="coerce"
)

# ------------------------------------------------------------
# 2. Create useful team-level variables
# ------------------------------------------------------------

team_match["team_win_flag"] = team_match["result"].map({
    "W": 1,
    "L": 0
})

team_match["team_margin"] = (
    team_match["team_score"] -
    team_match["opponent_score"]
)

# ------------------------------------------------------------
# 3. Calculate previous 5-game form
# ------------------------------------------------------------

team_match = team_match.sort_values(
    ["team_name", "match_date", "id"]
).reset_index(drop=True)

team_match["recent_form_5"] = (
    team_match
    .groupby("team_name")["team_win_flag"]
    .transform(
        lambda x: x.shift(1).rolling(
            5,
            min_periods=1
        ).mean()
    )
)

# ------------------------------------------------------------
# 4. Calculate days of rest
# ------------------------------------------------------------

team_match["days_rest"] = (
    team_match
    .groupby("team_name")["match_date"]
    .diff()
    .dt.days
)

# ============================================================
# VISUALIZATION 1
# Team win rate by season
# ============================================================

season_win_rate = (
    team_match
    .groupby("year")["team_win_flag"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(10, 5))
plt.plot(
    season_win_rate["year"],
    season_win_rate["team_win_flag"],
    marker="o"
)
plt.xlabel("Season")
plt.ylabel("Win Rate")
plt.title("AFL Team Win Rate by Season")
plt.grid(True, alpha=0.3)
plt.show()


# ============================================================
# VISUALIZATION 2
# Home vs Away win rate
# ============================================================

home_away_win_rate = (
    team_match
    .groupby("home_away")["team_win_flag"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(7, 5))
plt.bar(
    home_away_win_rate["home_away"],
    home_away_win_rate["team_win_flag"]
)
plt.xlabel("Home / Away")
plt.ylabel("Win Rate")
plt.title("Home vs Away Win Rate")
plt.ylim(0, 1)
plt.show()


# ============================================================
# VISUALIZATION 3
# Recent form vs win probability
# ============================================================

form_analysis = (
    team_match
    .dropna(subset=["recent_form_5", "team_win_flag"])
    .copy()
)

form_analysis["form_percentage"] = (
    form_analysis["recent_form_5"] * 100
)

form_bins = (
    form_analysis
    .groupby("form_percentage")["team_win_flag"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(9, 5))
plt.plot(
    form_bins["form_percentage"],
    form_bins["team_win_flag"],
    marker="o"
)
plt.xlabel("Previous 5-Game Win Rate (%)")
plt.ylabel("Current Match Win Rate")
plt.title("Recent Form vs Current Match Win Probability")
plt.grid(True, alpha=0.3)
plt.show()


# ============================================================
# VISUALIZATION 4
# Rest days vs win probability
# ============================================================

rest_analysis = (
    team_match
    .dropna(subset=["days_rest", "team_win_flag"])
    .copy()
)

# Remove extreme rest gaps for clearer visualization
rest_analysis = rest_analysis[
    rest_analysis["days_rest"].between(1, 30)
]

rest_analysis["rest_group"] = pd.cut(
    rest_analysis["days_rest"],
    bins=[0, 5, 7, 10, 14, 30],
    labels=[
        "1-5 days",
        "6-7 days",
        "8-10 days",
        "11-14 days",
        "15-30 days"
    ]
)

rest_win_rate = (
    rest_analysis
    .groupby("rest_group", observed=False)["team_win_flag"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(9, 5))
plt.bar(
    rest_win_rate["rest_group"].astype(str),
    rest_win_rate["team_win_flag"]
)
plt.xlabel("Days of Rest")
plt.ylabel("Win Rate")
plt.title("Rest Period vs Win Rate")
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.show()


# ============================================================
# VISUALIZATION 5
# Player goals distribution
# ============================================================

goals = pd.to_numeric(
    player_match["goals"],
    errors="coerce"
).dropna()

plt.figure(figsize=(9, 5))
plt.hist(goals, bins=20)
plt.xlabel("Goals per Game")
plt.ylabel("Number of Player-Games")
plt.title("Distribution of Player Goals per Game")
plt.show()


# ============================================================
# VISUALIZATION 6
# Player disposals distribution
# ============================================================

disposals = pd.to_numeric(
    player_match["disposals"],
    errors="coerce"
).dropna()

plt.figure(figsize=(9, 5))
plt.hist(disposals, bins=25)
plt.xlabel("Disposals per Game")
plt.ylabel("Number of Player-Games")
plt.title("Distribution of Player Disposals per Game")
plt.show()


# ============================================================
# VISUALIZATION 7
# Player fantasy points distribution
# ============================================================

fantasy = pd.to_numeric(
    player_match["fantasy_points"],
    errors="coerce"
).dropna()

plt.figure(figsize=(9, 5))
plt.hist(fantasy, bins=25)
plt.xlabel("Fantasy Points")
plt.ylabel("Number of Player-Games")
plt.title("Distribution of Player Fantasy Points")
plt.show()


# ============================================================
# HISTORICAL TOP PLAYERS
# ============================================================

# Top goal kickers
top_goal_kickers = (
    player_season
    .groupby("player_id", as_index=False)["goals"]
    .sum()
    .sort_values("goals", ascending=False)
    .head(10)
)

plt.figure(figsize=(10, 6))
plt.barh(
    top_goal_kickers["player_id"].astype(str),
    top_goal_kickers["goals"]
)
plt.xlabel("Total Goals")
plt.ylabel("Player ID")
plt.title("Historical Top 10 Goal Kickers")
plt.gca().invert_yaxis()
plt.show()


# Top disposal getters
top_disposal_getters = (
    player_season
    .groupby("player_id", as_index=False)["disposals"]
    .sum()
    .sort_values("disposals", ascending=False)
    .head(10)
)

plt.figure(figsize=(10, 6))
plt.barh(
    top_disposal_getters["player_id"].astype(str),
    top_disposal_getters["disposals"]
)
plt.xlabel("Total Disposals")
plt.ylabel("Player ID")
plt.title("Historical Top 10 Disposal Getters")
plt.gca().invert_yaxis()
plt.show()


# ============================================================
# EDA SUMMARY
# ============================================================

print("EDA completed successfully.")
print("Visualizations produced:")
print("1. Team win rate by season")
print("2. Home vs away win rate")
print("3. Recent form vs win probability")
print("4. Rest days vs win rate")
print("5. Goals distribution")
print("6. Disposals distribution")
print("7. Fantasy points distribution")
print("8. Historical top goal kickers")
print("9. Historical top disposal getters")

## Task 4: Feature Engineering for Prediction

### Rolling and form features
We will engineer pre-match team and player features using only information available before each match:

- team win streaks and moving win rates
- rolling average score for and against
- rolling player output for goals, disposals, fantasy points
- recent form windows (3-game, 5-game, 10-game) used as predictors only after shifting by one match

### Head-to-head and contextual features
We will include:
- historical head-to-head win rate between the two teams
- historical average margin in prior meetings
- days of rest from the prior match
- current ladder position / cumulative prior points before the match
- venue and home/away status

### Leakage control
All rolling and context features must be computed using only rows with `match_date < current_match_date`, never future matches. This keeps the feature table valid for anomaly detection, classification, and regression pipelines.

### Versioned feature store contract
The final feature table will be saved with a version tag and a feature dictionary containing:
- `feature_name`
- `description`
- `computation_window`
- `source_columns`

This dictionary becomes the contract for Day 2 model development and keeps the feature pipeline reproducible.


In [ ]:
# ============================================================
# TASK 4: FEATURE ENGINEERING
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Load datasets
# ------------------------------------------------------------

DATA_DIR = Path("afl_datasets")

if not DATA_DIR.exists():
    DATA_DIR = Path("Week 3/Day 1/afl_datasets")

team_match = pd.read_csv(
    DATA_DIR / "team_matches_home_away_raw - team_matches_home_away_raw.csv.csv",
    low_memory=False
)

player_match = pd.read_csv(
    DATA_DIR / "afl_players_round_by_round_stats_raw - afl_players_round_by_round_stats_raw.csv.csv",
    low_memory=False
)

# Convert dates
team_match["match_date"] = pd.to_datetime(
    team_match["match_date"],
    errors="coerce"
)

player_match["match_date"] = pd.to_datetime(
    player_match["match_date"],
    errors="coerce"
)

# Clean names
team_match["team_name"] = (
    team_match["team_name"]
    .astype("string")
    .str.strip()
)

team_match["opponent"] = (
    team_match["opponent"]
    .astype("string")
    .str.strip()
)

# ------------------------------------------------------------
# 2. Define match targets
# ------------------------------------------------------------

team_match["team_win_flag"] = team_match["result"].map({
    "W": 1,
    "L": 0
})

team_match["team_margin"] = (
    team_match["team_score"] -
    team_match["opponent_score"]
)

# Sort chronologically
team_match = team_match.sort_values(
    ["team_name", "match_date", "id"]
).reset_index(drop=True)


# ============================================================
# A. TEAM ROLLING / FORM FEATURES
# ============================================================

for window in [3, 5, 10]:

    # Previous win rate
    team_match[f"win_rate_last_{window}"] = (
        team_match
        .groupby("team_name")["team_win_flag"]
        .transform(
            lambda x:
            x.shift(1)
             .rolling(window, min_periods=1)
             .mean()
        )
    )

    # Average points scored previously
    team_match[f"avg_score_for_last_{window}"] = (
        team_match
        .groupby("team_name")["team_score"]
        .transform(
            lambda x:
            x.shift(1)
             .rolling(window, min_periods=1)
             .mean()
        )
    )

    # Average points conceded previously
    team_match[f"avg_score_against_last_{window}"] = (
        team_match
        .groupby("team_name")["opponent_score"]
        .transform(
            lambda x:
            x.shift(1)
             .rolling(window, min_periods=1)
             .mean()
        )
    )


# ============================================================
# B. WIN STREAK
# ============================================================

team_match = team_match.sort_values(
    ["team_name", "match_date", "id"]
).reset_index(drop=True)

win_streak = []
current_streak = {}

for _, row in team_match.iterrows():

    team = row["team_name"]

    # Streak BEFORE the current match
    win_streak.append(
        current_streak.get(team, 0)
    )

    # Update streak AFTER observing current result
    if row["result"] == "W":
        current_streak[team] = (
            current_streak.get(team, 0) + 1
        )
    else:
        current_streak[team] = 0

team_match["win_streak_before_match"] = win_streak


# ============================================================
# C. DAYS OF REST
# ============================================================

team_match["days_rest"] = (
    team_match
    .groupby("team_name")["match_date"]
    .diff()
    .dt.days
)


# ============================================================
# D. HEAD-TO-HEAD FEATURES
# ============================================================

# Create order-independent team pairing
team_match["pair_key"] = np.where(
    team_match["team_name"] < team_match["opponent"],
    team_match["team_name"] + "|" + team_match["opponent"],
    team_match["opponent"] + "|" + team_match["team_name"]
)

# Identify which side of the pair the current team represents
team_match["pair_first_team"] = (
    team_match["pair_key"]
    .str.split("|")
    .str[0]
)

team_match["pair_win"] = team_match["team_win_flag"]

team_match["pair_margin"] = (
    team_match["team_margin"] *
    np.where(
        team_match["team_name"]
        == team_match["pair_first_team"],
        1,
        -1
    )
)

team_match = team_match.sort_values(
    ["pair_key", "match_date", "id"]
).reset_index(drop=True)

# Historical H2H win rate
team_match["h2h_win_rate_before"] = (
    team_match
    .groupby("pair_key")["pair_win"]
    .transform(
        lambda x:
        x.shift(1)
         .expanding(min_periods=1)
         .mean()
    )
)

# Historical H2H margin
team_match["h2h_avg_margin_before"] = (
    team_match
    .groupby("pair_key")["pair_margin"]
    .transform(
        lambda x:
        x.shift(1)
         .expanding(min_periods=1)
         .mean()
    )
)


# ============================================================
# E. CUMULATIVE LADDER FEATURES
# ============================================================

team_match["match_points"] = team_match["result"].map({
    "W": 4,
    "D": 2,
    "L": 0
})

team_match = team_match.sort_values(
    ["match_date", "team_name", "id"]
).reset_index(drop=True)

# Points BEFORE current match
team_match["prior_points"] = (
    team_match
    .groupby("team_name")["match_points"]
    .cumsum()
    - team_match["match_points"]
)

# Scores accumulated BEFORE current match
team_match["prior_score_for"] = (
    team_match
    .groupby("team_name")["team_score"]
    .cumsum()
    - team_match["team_score"]
)

team_match["prior_score_against"] = (
    team_match
    .groupby("team_name")["opponent_score"]
    .cumsum()
    - team_match["opponent_score"]
)

# Historical percentage proxy
team_match["prior_percentage"] = np.where(
    team_match["prior_score_against"] > 0,
    100 *
    team_match["prior_score_for"] /
    team_match["prior_score_against"],
    np.nan
)

# Approximate ladder position based on prior points
team_match["ladder_position_before"] = (
    team_match
    .groupby(["year", "match_date"])["prior_points"]
    .rank(
        method="min",
        ascending=False
    )
)


# ============================================================
# F. PLAYER ROLLING FEATURES
# ============================================================

player_match = player_match.dropna(
    subset=["player_id", "match_date"]
).copy()

player_match = player_match.sort_values(
    ["player_id", "match_date", "id"]
).reset_index(drop=True)

for window in [3, 5, 10]:

    player_match[f"avg_goals_last_{window}"] = (
        player_match
        .groupby("player_id")["goals"]
        .transform(
            lambda x:
            x.shift(1)
             .rolling(window, min_periods=1)
             .mean()
        )
    )

    player_match[f"avg_disposals_last_{window}"] = (
        player_match
        .groupby("player_id")["disposals"]
        .transform(
            lambda x:
            x.shift(1)
             .rolling(window, min_periods=1)
             .mean()
        )
    )

    player_match[f"avg_fantasy_points_last_{window}"] = (
        player_match
        .groupby("player_id")["fantasy_points"]
        .transform(
            lambda x:
            x.shift(1)
             .rolling(window, min_periods=1)
             .mean()
        )
    )


# ============================================================
# G. ADD VERSION NUMBER
# ============================================================

team_match["feature_version"] = "v1.0"
player_match["feature_version"] = "v1.0"


# ============================================================
# H. SAVE VERSIONED FEATURE TABLES
# ============================================================

team_feature_file = Path("team_match_features_v1.csv")
player_feature_file = Path("player_match_features_v1.csv")

team_match.to_csv(
    team_feature_file,
    index=False
)

player_match.to_csv(
    player_feature_file,
    index=False
)


# ============================================================
# I. FEATURE DICTIONARY
# ============================================================

feature_dictionary = pd.DataFrame([
    {
        "feature_name": "win_rate_last_3",
        "description": "Team win rate over previous 3 matches",
        "computation_window": "Previous 3 matches",
        "source_columns": "result"
    },
    {
        "feature_name": "win_rate_last_5",
        "description": "Team win rate over previous 5 matches",
        "computation_window": "Previous 5 matches",
        "source_columns": "result"
    },
    {
        "feature_name": "win_rate_last_10",
        "description": "Team win rate over previous 10 matches",
        "computation_window": "Previous 10 matches",
        "source_columns": "result"
    },
    {
        "feature_name": "avg_score_for_last_3",
        "description": "Average team score in previous 3 matches",
        "computation_window": "Previous 3 matches",
        "source_columns": "team_score"
    },
    {
        "feature_name": "avg_score_for_last_5",
        "description": "Average team score in previous 5 matches",
        "computation_window": "Previous 5 matches",
        "source_columns": "team_score"
    },
    {
        "feature_name": "avg_score_for_last_10",
        "description": "Average team score in previous 10 matches",
        "computation_window": "Previous 10 matches",
        "source_columns": "team_score"
    },
    {
        "feature_name": "avg_score_against_last_3",
        "description": "Average opponent score in previous 3 matches",
        "computation_window": "Previous 3 matches",
        "source_columns": "opponent_score"
    },
    {
        "feature_name": "avg_score_against_last_5",
        "description": "Average opponent score in previous 5 matches",
        "computation_window": "Previous 5 matches",
        "source_columns": "opponent_score"
    },
    {
        "feature_name": "avg_score_against_last_10",
        "description": "Average opponent score in previous 10 matches",
        "computation_window": "Previous 10 matches",
        "source_columns": "opponent_score"
    },
    {
        "feature_name": "win_streak_before_match",
        "description": "Number of consecutive wins before current match",
        "computation_window": "All previous matches",
        "source_columns": "result"
    },
    {
        "feature_name": "days_rest",
        "description": "Days since team's previous match",
        "computation_window": "Previous match",
        "source_columns": "match_date"
    },
    {
        "feature_name": "h2h_win_rate_before",
        "description": "Team win rate in previous meetings against opponent",
        "computation_window": "All previous H2H meetings",
        "source_columns": "team_name, opponent, result"
    },
    {
        "feature_name": "h2h_avg_margin_before",
        "description": "Average team margin in previous H2H meetings",
        "computation_window": "All previous H2H meetings",
        "source_columns": "team_score, opponent_score"
    },
    {
        "feature_name": "prior_points",
        "description": "Cumulative ladder points before current match",
        "computation_window": "Season to date",
        "source_columns": "result"
    },
    {
        "feature_name": "ladder_position_before",
        "description": "Approximate ladder position based on prior points",
        "computation_window": "Season/date snapshot",
        "source_columns": "year, match_date, result"
    },
    {
        "feature_name": "home_away",
        "description": "Whether team is playing at home or away",
        "computation_window": "Current match",
        "source_columns": "home_away"
    },
    {
        "feature_name": "venue",
        "description": "Venue of current match",
        "computation_window": "Current match",
        "source_columns": "venue"
    },
    {
        "feature_name": "avg_goals_last_3",
        "description": "Player average goals over previous 3 games",
        "computation_window": "Previous 3 player games",
        "source_columns": "goals"
    },
    {
        "feature_name": "avg_disposals_last_3",
        "description": "Player average disposals over previous 3 games",
        "computation_window": "Previous 3 player games",
        "source_columns": "disposals"
    },
    {
        "feature_name": "avg_fantasy_points_last_3",
        "description": "Player average fantasy points over previous 3 games",
        "computation_window": "Previous 3 player games",
        "source_columns": "fantasy_points"
    },
    {
        "feature_name": "avg_goals_last_5",
        "description": "Player average goals over previous 5 games",
        "computation_window": "Previous 5 player games",
        "source_columns": "goals"
    },
    {
        "feature_name": "avg_disposals_last_5",
        "description": "Player average disposals over previous 5 games",
        "computation_window": "Previous 5 player games",
        "source_columns": "disposals"
    },
    {
        "feature_name": "avg_fantasy_points_last_5",
        "description": "Player average fantasy points over previous 5 games",
        "computation_window": "Previous 5 player games",
        "source_columns": "fantasy_points"
    },
    {
        "feature_name": "avg_goals_last_10",
        "description": "Player average goals over previous 10 games",
        "computation_window": "Previous 10 player games",
        "source_columns": "goals"
    },
    {
        "feature_name": "avg_disposals_last_10",
        "description": "Player average disposals over previous 10 games",
        "computation_window": "Previous 10 player games",
        "source_columns": "disposals"
    },
    {
        "feature_name": "avg_fantasy_points_last_10",
        "description": "Player average fantasy points over previous 10 games",
        "computation_window": "Previous 10 player games",
        "source_columns": "fantasy_points"
    }
])

feature_dictionary["feature_version"] = "v1.0"

feature_dictionary.to_csv(
    "feature_dictionary_v1.csv",
    index=False
)


# ============================================================
# J. VALIDATION
# ============================================================

print("TASK 4 COMPLETED")
print("=" * 60)

print("\nTeam feature table:")
print(team_match.shape)

print("\nPlayer feature table:")
print(player_match.shape)

print("\nFeature dictionary:")
print(feature_dictionary.shape)

print("\nSaved files:")
print("- team_match_features_v1.csv")
print("- player_match_features_v1.csv")
print("- feature_dictionary_v1.csv")

print("\nTeam features:")
print([
    col for col in team_match.columns
    if col in [
        "win_rate_last_3",
        "win_rate_last_5",
        "win_rate_last_10",
        "avg_score_for_last_3",
        "avg_score_for_last_5",
        "avg_score_for_last_10",
        "win_streak_before_match",
        "days_rest",
        "h2h_win_rate_before",
        "h2h_avg_margin_before",
        "prior_points",
        "ladder_position_before"
    ]
])

print("\nLeakage control:")
print(
    "All rolling and historical features use shift(1), "
    "so the current match result is excluded from its predictors."
)

## Task 5: Reproducible Train/Hold-Out Split

### Time-based split strategy

We will use a strict chronological split rather than a random train/test split. The most recent season in the dataset will be held out as the final test set, while all earlier seasons will be used for training. This reflects the real prediction setting: when predicting a future AFL match, only information from previous matches and seasons should be available to the model.

A random split could leak future information because matches from the same season—and potentially later rounds—could appear in both the training and test sets. This would allow the model to learn patterns from future games that would not actually be available at prediction time. A time-based split therefore gives a more realistic estimate of how the model should perform on genuinely unseen future matches.

### Split rule

* **Training set:** all observations from seasons before the most recent season.
* **Hold-out/test set:** all observations from the most recent season.
* The split is determined from the chronological `year` field rather than randomly sampling rows.
* The same split function will be reused by all models developed during the week.

### Realistic prediction ceiling

A very high prediction accuracy should not be expected because AFL matches are inherently noisy and affected by injuries, form, tactics, weather, umpiring, and random events. A useful model should perform meaningfully better than simple baselines while still leaving room for genuine uncertainty. Perfect or near-perfect accuracy would be a major warning sign because real sporting outcomes contain substantial unpredictability. Such performance would therefore suggest possible data leakage, such as future match results or post-match statistics being included in the predictors.


In [ ]:
def time_based_split(df, year_col="year"):
    """
    Chronological train/hold-out split.

    Training data:
        All seasons before the most recent season.

    Hold-out data:
        The most recent season.

    Returns:
        train_df, holdout_df, train_seasons, holdout_season
    """
    data = df.copy()

    # Remove rows without a valid season
    data = data.dropna(subset=[year_col])

    # Sort chronologically
    data = data.sort_values(year_col).reset_index(drop=True)

    # Identify the most recent season
    seasons = sorted(data[year_col].unique())
    holdout_season = seasons[-1]

    # Strict chronological split
    train_df = data[data[year_col] < holdout_season].copy()
    holdout_df = data[data[year_col] == holdout_season].copy()

    return train_df, holdout_df, seasons[:-1], holdout_season


# Example: split the team-match dataset
train_team, holdout_team, train_seasons, holdout_season = time_based_split(
    frames["team_match"],
    year_col="year"
)

print("Training seasons:", train_seasons)
print("Hold-out season:", holdout_season)
print("Training rows:", len(train_team))
print("Hold-out rows:", len(holdout_team))

Training seasons: [np.int64(1983), np.int64(1984), np.int64(1985), np.int64(1986), np.int64(1987), np.int64(1988), np.int64(1989), np.int64(1990), np.int64(1991), np.int64(1992), np.int64(1993), np.int64(1994), np.int64(1995), np.int64(1996), np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Hold-out season: 2025
Training rows: 15376
Hold-out rows: 432
